# Train Canonical Evaluator Adapter: `SANCTIONS_SCREENING_MATCH` (v4 Multi-Format)
**Telegraph Protocol Evaluator Adapter for ModernBERT-base Foundation Model**
- Intent: `SANCTIONS_SCREENING_MATCH`
- Class: `hybrid`
- Foundation: `modernbert-base-lora16-v1`
- Dataset: `eval-data/pairs_sanctions_v4.jsonl` (1,000 multi-format pairs, 0 entity overlap across splits, 57.6% grouped Naive Bayes surface probe)
- LoRA Config: rank 8, alpha 32, target modules `Wqkv, Wo, Wi`

In [ ]:
# 1. Setup Environment, Baseline Repo & Foundation Assets
!pip install -q transformers>=4.48 peft datasets onnx onnxruntime safetensors
!rm -rf baseline && git clone https://github.com/telegraphprotocol/telegraph-wasm-baseline.git baseline
%cd baseline
!mkdir -p foundation
!curl -sSL https://github.com/telegraphprotocol/telegraph-foundation/releases/download/modernbert-base-lora16-v1/foundation.json -o foundation/foundation.json
!curl -sSL https://github.com/telegraphprotocol/telegraph-foundation/releases/download/modernbert-base-lora16-v1/tokenizer.json -o foundation/tokenizer.json
!curl -sSL https://github.com/telegraphprotocol/telegraph-foundation/releases/download/modernbert-base-lora16-v1/foundation_engine.wasm -o foundation/foundation_engine.wasm
!curl -sSL https://github.com/telegraphprotocol/telegraph-foundation/releases/download/modernbert-base-lora16-v1/foundation_lora_fp32.onnx -o foundation/foundation_lora_fp32.onnx
!sha256sum foundation/*

In [ ]:
# 2. Download v4 Multi-Format Sanctions Dataset & Materialize Disjoint Train/Val Splits
import json

!curl -sSL "https://raw.githubusercontent.com/VishAlgoDev/praman-desk/main/eval-data/pairs_sanctions_v4.jsonl?v=$(date +%s)" -o pairs_sanctions_v4.jsonl

train_rows, val_rows = [], []
with open('pairs_sanctions_v4.jsonl', 'r', encoding='utf-8') as f:
  for line in f:
    row = json.loads(line)
    pair = {'prompt': row['prompt'], 'chosen': row['chosen'], 'rejected': row['rejected']}
    if row.get('split') == 'val':
      val_rows.append(pair)
    else:
      train_rows.append(pair)

with open('pairs_v4_train.jsonl', 'w', encoding='utf-8') as f:
  for r in train_rows: f.write(json.dumps(r) + '\n')
with open('pairs_v4_val.jsonl', 'w', encoding='utf-8') as f:
  for r in val_rows: f.write(json.dumps(r) + '\n')

print(f'Loaded {len(train_rows)} train + {len(val_rows)} val pairs (disjoint entities)')

In [ ]:
# 3. Run Official Anti-Shortcut Dataset Gate Check
!python -m float_runtime.train_adapter \
  --intent SANCTIONS_SCREENING_MATCH \
  --data pairs_v4_train.jsonl \
  --val-data pairs_v4_val.jsonl \
  --out runs/SANCTIONS_SCREENING_MATCH_v4 \
  --check-data

In [ ]:
# 4. Train LoRA Evaluator Adapter (v4 Disjoint Train/Val)
import torch
if torch.cuda.is_available():
  print('GPU Available:', torch.cuda.get_device_name(0))
else:
  print('No GPU available / Colab GPU limit reached. Training on CPU in fp32...')

!python -m float_runtime.train_adapter \
  --intent SANCTIONS_SCREENING_MATCH \
  --data pairs_v4_train.jsonl \
  --val-data pairs_v4_val.jsonl \
  --out runs/SANCTIONS_SCREENING_MATCH_v4 \
  --lora-rank 8 \
  --lora-alpha 32 \
  --epochs 4 \
  --batch-size 8 \
  --lr 5e-4

In [ ]:
# 5. Export Canonical LoRA safetensors Delta Package (class: hybrid)
!python -m float_runtime.export_adapter \
  --run runs/SANCTIONS_SCREENING_MATCH_v4 \
  --intent SANCTIONS_SCREENING_MATCH \
  --class hybrid \
  --foundation foundation/foundation.json

!ls -lh runs/SANCTIONS_SCREENING_MATCH_v4

In [ ]:
# 6. Official Go Stage-1 Validator (adapterbuild lora) + Download Artifacts
import shutil, json
if not shutil.which('go'):
  !curl -sSL https://go.dev/dl/go1.25.0.linux-amd64.tar.gz -o /tmp/go.tgz && rm -rf /usr/local/go && tar -C /usr/local -xzf /tmp/go.tgz
  import os; os.environ['PATH'] = '/usr/local/go/bin:' + os.environ['PATH']

!go build -o adapterbuild ./cmd/adapterbuild
!./adapterbuild lora \
  --intent SANCTIONS_SCREENING_MATCH \
  --package runs/SANCTIONS_SCREENING_MATCH_v4/SANCTIONS_SCREENING_MATCH.safetensors \
  --foundation foundation/foundation.json \
  --out runs/SANCTIONS_SCREENING_MATCH_v4 \
  --class hybrid \
  --foundation-onnx foundation/foundation_lora_fp32.onnx \
  --engine foundation/foundation_engine.wasm | tee runs/SANCTIONS_SCREENING_MATCH_v4/validator.json

with open('runs/SANCTIONS_SCREENING_MATCH_v4/result.json') as f:
  print(json.dumps(json.load(f), indent=2))

from google.colab import files
files.download('runs/SANCTIONS_SCREENING_MATCH_v4/SANCTIONS_SCREENING_MATCH.safetensors')
files.download('runs/SANCTIONS_SCREENING_MATCH_v4/result.json')
files.download('runs/SANCTIONS_SCREENING_MATCH_v4/validator.json')